# T42SUJ — 2025 aprel, Qashqadaryo: kirish ma'lumotlari va SEBAL (ET gacha)

**Ierarxiya:** hudud (viloyat) → uni qoplovchi MGRS tile'lar → tile ichidagi ekin. Hudud — Qashqadaryo (`uzb_admin1_2026` asseti), tile — T42SUJ.
- Tasvir qoidalari — **tile ∩ Qashqadaryo ekinida**;
- anchor (va z0m persentillari, sahna QC) ham **tile ∩ Qashqadaryo**da — tile Samarqandni qoplasa ham anchor u yerga ketmaydi;
- VIIRS RF downscaling o'qitish — butun tile (ko'proq namuna).

Ko'p sensorli yo'l (`mode_process='LHLSVIIRSECO'`). Hamma hisob **mavjud pipeline funksiyalari** orqali — notebook faqat chaqiradi va ko'rsatadi:

| Qadam | Pipeline funksiyasi |
|---|---|
| Viloyat → tile'lar (avtomatik): topish, kichiklarini tushirish, ustma-ustlikni bo'lish | `pipeline.tile.mgrs_tiles_for` → `pipeline.tile.plan_tiles` |
| Tile → ekin: qaysi WRS sahnasi qaysi ekinni ko'radi | `pipeline.calendar.TileCalendar.wrs_coverage` |
| Kirish bosqichi (qaysi tasvir qabul, qaysi biri nega rad) | `pipeline.calendar.TileCalendar(region=...)` |
| Landsat sahnasi (QA, scale, burchak, ERA5) → SMW LST | `pipeline.scenes.landsat_scene` → `surface_props.compute_all` → `radiation.compute_all` |
| VIIRS LST downscaling'dan oldin / keyin | `pipeline.scenes.viirs_coarse_lst` / `pipeline.scenes.viirs_scene` |
| SEBAL: anchor kaskadi → energiya balansi → ET_24 | `pipeline.multisensor.process_tile` → `main.process_scene` |

Xaritalar: Google Hybrid asos xarita; binafsha — tile chegarasi, havorang — Qashqadaryo chegarasi. LST palitrasi — **cividis**, hamma LST qatlamlarida bir xil oraliq (aprel: 295–315 K — ma'lumot shu oraliqda; kengroq oraliqda cividis'ning kulrang o'rtasi "chang"dek ko'rinadi). Ko'p qatlamlar sukut bo'yicha yashirin — o'ngdagi qatlam boshqaruvidan yoqing.

Kernel: `gee_env`. GEE loyihasi: `ee-chexovant11`. Hisob: reja ~1 daqiqa, kalendar ~1–3 daqiqa, SEBAL ~2–3 daqiqa har sahna. VIIRS 30 m (RF) qatlami xaritada har safar GEE'da hisoblanadi — sekin chizilishi mumkin.

In [ ]:
import os, sys
SCRIPTS = r'D:\Cloud_comp\Sebal\scripts'
sys.path.insert(0, SCRIPTS)

import ee
ee.Initialize(project='ee-chexovant11')

import numpy as np
import pandas as pd
import matplotlib
import folium
import geemap.foliumap as geemap
from IPython.display import display

from sebal_gee_v4 import config as cfg, ee_utils, radiation, surface_props, daily_et
from sebal_gee_v4.pipeline import multisensor, scenes
from sebal_gee_v4.pipeline.calendar import TileCalendar
from sebal_gee_v4.pipeline.tile import MgrsTile, plan_tiles, mgrs_tiles_for
from sebal_gee_v4.pipeline.rules import InputRules, ESRI_LULC, ESRI_CROPS, ADMIN1, ADMIN1_FIELD

ee_utils.install_getinfo_retry()

REGION_NAME = 'Qashqadaryo'                 # uzb_admin1_2026: region_nam
TILE_ID = 'T42SUJ'
START, END = '2025-04-01', '2025-05-01'      # END kirmaydi
MODE = 'SEBAL_Milliy'
RULES = InputRules()                        # kelishilgan qoidalar: bulut 80% / 20% / 10%, VIIRS QC, anchor H3

adm = ee.FeatureCollection(ADMIN1)
region = cfg.build_roi('asset', asset_id=ADMIN1, name=REGION_NAME, name_field=ADMIN1_FIELD)


def palette(name, n=11):
    return [matplotlib.colors.to_hex(c) for c in matplotlib.colormaps[name](np.linspace(0, 1, n))]


LST_VIS = {'min': 295, 'max': 315, 'palette': palette('cividis')}     # K — aprel (p2–p98 ≈ 296–311 K)
RGB_VIS = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}
ET_VIS = {'min': 0, 'max': 7, 'palette': palette('YlGnBu')}           # mm/kun
GOOGLE_HYBRID = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}'


def new_map(center, zoom):
    m = geemap.Map(basemap=None)             # geemap 0.36: basemap='HYBRID' = Esri, Google emas
    m.add_tile_layer(GOOGLE_HYBRID, name='Google Hybrid', attribution='Google', overlay=False)
    m.centerObject(center, zoom)
    return m


def outline(geom, color, width=2, dashed=False):
    st = {'color': color, 'fillColor': '00000000', 'width': width}
    if dashed:
        st['lineType'] = 'dashed'
    return ee.FeatureCollection([ee.Feature(geom)]).style(**st)


def label(m, lonlat, text, color):
    folium.Marker([lonlat[1], lonlat[0]], icon=folium.DivIcon(icon_size=(170, 36), html=(
        f"<div style='font:bold 13px sans-serif;color:{color};text-shadow:0 0 3px #000;white-space:nowrap'>"
        f"{text}</div>"))).add_to(m)


def add_frame(m, tile):
    m.addLayer(outline(region, '00ffff', 3), {}, f'{REGION_NAME} chegarasi')
    m.addLayer(outline(tile.geometry, 'ff00ff', 3), {}, f'{TILE_ID} chegarasi')

## 0. Viloyat → tile'lar: avtomatik reja (har viloyat uchun bir xil qoida)
`main.run(..., tiles=None)` ham aynan shunday ishlaydi:
1. viloyatni qoplovchi MGRS tile'lar HLS granulalaridan topiladi;
2. tile'ning to'liq footprint'ida viloyat ekini < `min_calib_crop_km2` (vaqtincha 100 km²) bo'lsa — tushiriladi (anchor ishonchsiz);
3. ustma-ustlik: tile'lar viloyat ekini ko'pligi bo'yicha tartiblanadi — birinchisi **butun**, keyingisi faqat qolgan joyni (**kesik**) oladi; har piksel bitta tile'da;
4. kalibratsiya (kalendar, anchor) — har tile'ning butun footprint'i ∩ viloyat; eksport — faqat egaligi.

Xaritada har rang — bitta tile egaligi (raqam — tartib).

In [ ]:
REGION_TILES = mgrs_tiles_for(region, int(START[:4]))         # viloyat tile'lari — avtomatik
plan, plan_sum, owned = plan_tiles(REGION_TILES, region, int(START[:4]), RULES.min_calib_crop_km2)
display(pd.DataFrame(plan))
print(plan_sum)

COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
mp = new_map(region, 8)
ey = MgrsTile(TILE_ID).esri_year(int(START[:4]))
mp.addLayer(ee.ImageCollection(ESRI_LULC).filterBounds(region).filterDate(f'{ey}-01-01', f'{ey + 1}-01-01')
            .mosaic().select(0).eq(ESRI_CROPS).selfMask().clip(region), {'palette': ['ffffff']},
            f'{REGION_NAME} ekini (Esri {ey})')
keep = [r for r in plan if r['ishlanadi']]
parts = [owned[r['tile']].intersection(region, ee.ErrorMargin(100)) for r in keep]
cen = ee.List([g.centroid(100).coordinates() for g in parts]).getInfo()
for k, (r, g, c) in enumerate(zip(keep, parts, cen)):
    col = COLORS[k % len(COLORS)]
    mp.addLayer(ee.FeatureCollection([ee.Feature(g)]).style(color=col[1:], fillColor=col[1:] + '55', width=2), {},
                f"{r['tartib']}. {r['tile']} — {r['holat']}")
    label(mp, c, f"{r['tartib']}. {r['tile']}<br>{r['ekin_egalik_km2']:.0f} km²", '#ffffff')
for r in plan:
    if not r['ishlanadi']:
        mp.addLayer(outline(MgrsTile(r['tile']).geometry, 'ff3030', 2, dashed=True), {}, f"{r['tile']} — {r['holat']}")
mp.addLayer(outline(region, '00ffff', 3), {}, f'{REGION_NAME} chegarasi')
mp.add_layer_control()
mp

## 1. Tile → ekin: qaysi WRS sahnasi bizga foydali
Tile'ga bir nechta WRS-2 sahnasi tegadi, lekin sahna **viloyat ekinini** ko'rsagina foydali. Jadvallar:
- ekin maydoni — tile, tile ∩ Qashqadaryo va tile ichidagi har viloyat (Esri 10 m, sinf 5; asset faqat O'zbekiston — Tojikiston qismi jadvalda yo'q);
- har WRS sahnasi footprint'i (bulutdan qat'i nazar) shu ekinlarning necha foizini ko'radi.

Xaritada yashil — Qashqadaryo ekinini ko'radigan (foydali) sahna, qizil uzuq chiziq — ko'rmaydigan sahna. Sariq — tile ekini, to'q sariq — tile ∩ Qashqadaryo ekini. Qashqadaryo ekinini ko'rmaydigan sahnalar kalendarda o'zi rad etiladi (ekinning ko'rilgan ulushi < 10%).

In [ ]:
cal = TileCalendar(TILE_ID, START, END, RULES, region=region)      # qoidalar: tile ∩ viloyat ekini
tile = cal.tile
YEAR = int(START[:4])

near = adm.filterBounds(tile.geometry)
names = near.aggregate_array(ADMIN1_FIELD).getInfo()
regions = {n: near.filter(ee.Filter.eq(ADMIN1_FIELD, n)).geometry() for n in names}

wrs = cal.wrs_coverage(regions)
print(f'Ekin maydoni (Esri {tile.esri_year(YEAR)}):')
display(cal.crop_areas)
print("WRS sahnalari — footprint ekinning necha foizini ko'radi:")
display(wrs)

USEFUL = 'tile ∩ hudud ekinidan_%'
m0 = new_map(tile.geometry, 8)
crop = tile.lulc(YEAR).eq(ESRI_CROPS).selfMask()      # Esri 10 m xom sinf (displey uchun tez)
m0.addLayer(crop.clip(tile.geometry), {'palette': ['ffdb5c']}, f'Esri ekin (sinf 5, {tile.esri_year(YEAR)}) — butun tile')
m0.addLayer(crop.clip(cal.geom), {'palette': ['ff7f0e']}, f'Esri ekin — tile ∩ {REGION_NAME}')
for n, g in regions.items():
    m0.addLayer(outline(g, 'ffffff', 1), {}, f'viloyat: {n}')
centers = ee.List([cal.footprints[w].centroid(100).coordinates() for w in wrs['WRS']]).getInfo()
for (_, r), c in zip(wrs.iterrows(), centers):
    ok = r[USEFUL] >= RULES.crop_seen_min
    color = '#00ff00' if ok else '#ff3030'
    m0.addLayer(outline(cal.footprints[r['WRS']], color[1:], 2, dashed=not ok), {},
                f"WRS {r['WRS']} — {'foydali' if ok else 'foydasiz'} ({REGION_NAME} ekinining {r[USEFUL]:.0f}%)")
    label(m0, c, f"{r['WRS']}<br>{REGION_NAME} ekini {r[USEFUL]:.0f}%", color)
add_frame(m0, tile)
m0.add_layer_control()
m0

## 2. Kirish bosqichi — kalendar (qoidalar tile ∩ Qashqadaryo ekinida)
Har tasvir tile bo'yicha kesiladi; qoidalar: dastlabki bulut < 80%, tasvir ko'rgan (tile ∩ Qashqadaryo) ekin ustida bulut ≤ 20%, tasvir tile ∩ Qashqadaryo ekinining ≥ 10% ini ko'rishi shart. VIIRS: QC toza, |VZA| ≤ 40°, HLS optika ±2 kun (piksel ikkalasida ham toza).

In [ ]:
cal.run()
T = cal.tables()

print('Oylik xulosa:')
display(cal.summary)
print('Landsat — aprelning hamma tasvirlari:')
display(T['Landsat'].reindex(columns=['sana', 'sun_yoldosh', 'path', 'qatorlar', 'UTC', "ekin_ko'rildi_%",
                                      'ekin_bulut_%', 'qabul', 'sabab']))
print('VIIRS — qabul qilingan kunlar va HLS juftligi:')
V = T['VIIRS']
display(V[V['qabul'] == True].reindex(columns=['sana', 'VZA_median', 'View_Time', "ekin_ko'rildi_%",
                                               'ekin_bulut_%', 'juft', 'HLS', 'HLS_sana', 'HLS_siljish', 'juft_sabab']))
print('SEBAL kunlari:')
display(cal.days[cal.days['kun_turi'] != 'SB'])

## 3. Xarita: aprelning Landsat tasvirlari (har o'tish alohida)
Har Landsat kuzatuvi (bir kunlik qatorlar mozaikasi) uchun ikki qatlam: RGB (bulut niqoblangan) va SMW LST. Rad etilganlari ham ko'rsatiladi (nomida sababi); metadata filtridan (sahna buluti ≥ 80%) o'tmaganlarini qurib bo'lmaydi — ular faqat jadvalda.

In [ ]:
m1 = new_map(tile.geometry, 9)
first = True
for o in cal.obs['Landsat']:
    if not o['_refs']:
        continue
    img = scenes.landsat_scene(o, tile.geometry)
    img = radiation.compute_all(surface_props.compute_all(img, MODE, tile.geometry), MODE)   # LST = SMW
    tag = (f"{o['sana']} {o['sun_yoldosh']} p{o['path']} r{o['qatorlar']} "
           f"({'qabul' if o['qabul'] else 'rad: ' + o['sabab']})")
    m1.addLayer(img, RGB_VIS, f'Landsat RGB {tag}', False)
    m1.addLayer(img.select('LST'), LST_VIS, f'Landsat LST {tag}', first and o['qabul'])
    first = first and not o['qabul']
add_frame(m1, tile)
m1.add_colorbar(LST_VIS, label='LST, K (cividis)')
m1.add_layer_control()
m1

## 4. Xarita: VIIRS LST — downscaling'dan oldin (1 km) va keyin (30 m, RF)
Oldin: toza piksellar, tile'ning UTM 1 km gridiga bilinear. Keyin: HLS optika (kalendar tanlagan L30/S30) bilan Random Forest + energiya saqlanishi. 30 m qatlamda dala/qishloq/daryo naqshi ko'rinadi — RF LST'ni optik bandlar (NDVI, albedo, …) orqali tiklaydi, bu kutilgan hol (chang emas). Oraliq va palitra Landsat LST bilan bir xil.

In [ ]:
m2 = new_map(tile.geometry, 9)
for v in cal.obs['VIIRS']:
    if not v.get('juft'):
        continue
    h = v['_hls']
    tag = f"{v['sana']} (VZA {v['VZA_median']:.0f}°, optika {h['sensor']} {h['sana']})"
    m2.addLayer(scenes.hls_optics(h), RGB_VIS, f'HLS optika {tag}', False)
    m2.addLayer(scenes.viirs_coarse_lst(tile, v, RULES), LST_VIS, f'VIIRS LST 1 km — oldin {tag}', False)
    m2.addLayer(scenes.viirs_scene(tile, v, h, tile.geometry, RULES).select('LST'), LST_VIS,
                f'VIIRS LST 30 m — keyin {tag}', False)
add_frame(m2, tile)
m2.add_colorbar(LST_VIS, label='LST, K (cividis)')
m2.add_layer_control()
m2

## 5. SEBAL — har sahna: anchor kaskadi → energiya balansi → ET_24
Sahnalar — kalendar qabul qilganlari. **Kalibratsiya hududi (anchor, z0m, QC) — tile ∩ Qashqadaryo** (kalendardan olinadi). Quyidagi log — pipeline'ning o'z chiqishi: har sahnada anchor qaysi metod/zona bilan qidirilgani (↪ topilmadi, ✅ topildi), hot suv balansi, energiya balansi iteratsiyasi va QC ogohlantirishlari.

In [ ]:
roi = tile.geometry                         # sahna yig'uvchi / RF — butun MGRS tile
details = []
daily, info = multisensor.process_tile(
    roi, START, END, MODE, TILE_ID,
    anchor_method='cimec', anchor_mode='point_anchor',     # SEBAL_Milliy — main.process_tile ham point_anchor qiladi
    ref_type='alfalfa', utc_offset=5, etr24_source='era5',          # UZT = UTC+5 (avtomatik boylam/15 bu yerda +4 beradi)
    sloping_terrain=False, z_ws=0.0, prefix='  ',
    rules=RULES, calendar=cal, details=details, region_label=REGION_NAME)
print('Kunlik tasvirlar:', list(zip(info['dates'], info['solar_src'])))

## 6. Anchor tartibi — hot va cold qaysi tartibda topildi
Har sahna uchun kaskad urinishlari ketma-ket: metod (cimec → plan_a → plan_b → default → pysebal) va zona (H3: H2 — ekin ichidagi yalang'och dala, H1 — Esri 8, aralash).

In [ ]:
rows = []
for dd in details:
    for k, u in enumerate(dd['qc'].get('_urinishlar', []), 1):
        rows.append({'sana': dd['sana'], 'sahna': dd['tur'], 'tartib': k, 'metod': u['metod'], 'zona': u['zona'],
                     'natija': 'topildi' if u['topildi'] else 'topilmadi',
                     'cold_LST': u['cold_LST'] if (u['cold_LST'] or -999) > 0 else None,
                     'hot_LST': u['hot_LST'] if (u['hot_LST'] or -999) > 0 else None,
                     'sabab': u['sabab']})
display(pd.DataFrame(rows))

## 7. Xarita: ET_24, cold/hot nomzod maydonlari va tanlangan nuqtalar
Ko'k — cold nomzodlar va cold nuqta, qizil — hot nomzodlar va hot nuqta. Marker ustiga kursor qo'ysangiz metod/zona ko'rinadi.

In [ ]:
m3 = new_map(cal.geom, 9)
for dd in details:
    qc, out, a = dd['qc'], dd['natija'], dd['qc'].get('_anchors') or {}
    tag = f"{dd['sana']} {dd['tur']}"
    if out is not None:
        m3.addLayer(out.select('ET_24'), ET_VIS, f'ET_24 {tag}', False)
    if a.get('cold_mask') is not None:
        m3.addLayer(a['cold_mask'].selfMask(), {'palette': ['00a0ff']}, f'cold nomzodlar {tag}', False)
    if a.get('hot_mask') is not None:
        m3.addLayer(a['hot_mask'].selfMask(), {'palette': ['ff3000']}, f'hot nomzodlar {tag}', False)
    for side, color in (('cold', 'blue'), ('hot', 'red')):
        pt = a.get(f'{side}_point')
        if pt and None not in pt:
            m3.add_marker(location=[pt[1], pt[0]], icon=folium.Icon(color=color),
                          tooltip=f"{side} — {tag}: {a.get('method')}/{a.get('zone')}")
add_frame(m3, tile)
m3.add_colorbar(ET_VIS, label='ET_24, mm/kun')
m3.add_layer_control()
m3

## 8. Jadval: har sahna — anchor (qaysi viloyatda), cold/hot, ET (tile ∩ Qashqadaryo ekinida, 300 m)

In [ ]:
crop300 = tile.crop_fraction(YEAR, 300).gte(0.5)
rows, pts = [], []
for dd in details:
    qc, out, a = dd['qc'], dd['natija'], dd['qc'].get('_anchors') or {}
    r = {'sana': dd['sana'], 'sahna': dd['tur'], 'optika': qc.get('optika'), 'anchor': qc.get('anchor'),
         'holat': qc.get('status'), 'cold_LST': qc.get('cold_LST'), 'hot_LST': qc.get('hot_LST'),
         'cold_nuqta': a.get('cold_point'), 'hot_nuqta': a.get('hot_point'), 'sabab': qc.get('sabab')}
    for side in ('cold', 'hot'):
        pt = a.get(f'{side}_point')
        if pt and None not in pt:
            pts.append((len(rows), f'{side}_viloyat', adm.filterBounds(ee.Geometry.Point(pt)).aggregate_array(ADMIN1_FIELD)))
    if out is not None:
        st = out.select('ET_24').updateMask(crop300).reduceRegion(
            ee.Reducer.percentile([10, 50, 90]), cal.geom, crs=tile.crs,
            crsTransform=tile.transform(300), maxPixels=1e9).getInfo()
        r.update({'ET_p10': st.get('ET_24_p10'), 'ET_p50': st.get('ET_24_p50'), 'ET_p90': st.get('ET_24_p90')})
    rows.append(r)
for (i, key, _), v in zip(pts, ee.List([p[2] for p in pts]).getInfo() if pts else []):
    rows[i][key] = ', '.join(v) or "viloyatdan tashqarida"
display(pd.DataFrame(rows).round(2))